# Joint move

Move the arm to a joint configuration, then report the measured joints and their
error from the target while holding. This mirrors `joint_move.py`.

Ensure the robot is clear before running motion commands.

In [ ]:
import time

import numpy as np

from r2_labs import client, rpc_api

np.set_printoptions(precision=4, floatmode="fixed", suppress=True, linewidth=100)

host = "localhost"
server_address = f"tcp://{host}:{rpc_api.DEFAULT_PORT}"
query_address = f"tcp://{host}:{rpc_api.DEFAULT_QUERY_PORT}"
training_address = f"tcp://{host}:{rpc_api.DEFAULT_MODEL_TRAINER_PORT}"

robot = client.Robot(
    server_address,
    query_server_address=query_address,
    training_server_address=training_address,
)

In [ ]:
target = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0], dtype=np.float32)
hold_seconds = 5

In [ ]:
robot.exec_mode.set_execution_mode(rpc_api.ExecutionMode.READY)

print("Moving ...")
robot.behaviour.go_to_joints(configuration=target).result()

print(f"Target joints:   {target}")
for _ in range(hold_seconds):
  measured = robot.raw_robot.get_proprio_data().joint_positions
  assert measured is not None
  report = f"Measured joints: {measured}"
  if measured.shape == target.shape:
    max_err = float(np.max(np.abs(measured - target)))
    report += f"  max err: {max_err:.4f} rad ({np.degrees(max_err):.2f} deg)"
  print(report)
  time.sleep(1.0)

Optional: stop the robot when you are done.


In [ ]:
robot.exec_mode.set_execution_mode(rpc_api.ExecutionMode.STOP)
